# EMERGING-VALUE — 5-Year Hold Horizon

**Same strategy as `strategy_emerging_value.ipynb`** — identical gates, identical opportunity-score weights. The ONLY difference is the performance-evaluation horizon: every metric is computed over a **5-year hold** instead of 3-year.

## Why a 5yr variant?

- **Match private-MF investor reality**: sub-institutional buyers typically hold 5-7 years, not 3.
- **Wash out mid-cycle noise**: a 3yr hold entered in 2019 exits mid-2022 rate shock (bad timing). Same entry with 5yr hold exits into 2024 normalization.
- **Test yield-anchor thesis on longer time**: VALUE's cap-spread + yield tilt should compound better over 5yr than momentum plays.

## Data availability

- `zori_forward_return_60mo` observable **2015-2020** (6 cohorts, ZORI window binds)
- `hpi_forward_return_60mo` observable 1975-2020 (long history)
- Ex-ante (no 5yr forward yet): **2021-2025** (5 cohorts)
- Result: 6 observable × 10 picks = **60 scored picks** (vs 80 for 3yr)

## Historical performance (2015-2020 cohorts, 5yr forward)

- Above-avg hit rate: **81.7%** (vs 3yr's 75.0% — longer hold improves consistency)
- Mean edge: **+14.91pp** (vs 3yr's +8.20pp — compounding advantage)
- Underwater picks: **0/60** (vs 3yr's 0/80 — same margin of safety)
- Worst pick: McHenry IL 2016, +42.93% total (+7.4%/yr) — even the worst historical entry beat trend HPI


## EMERGING-VALUE — Criteria (unchanged from 3yr notebook)

Strategy is identical to `strategy_emerging_value.ipynb`. See that notebook or this section's summary below.

### Layer 1 — Universe
| Filter | Value | Purpose |
|---|---|---|
| Population | 300K – 1M | Mid-size metros — below institutional radar |
| Employment diversity HHI | ≤ 0.20 | Broadened universe (strategy HHI ≤ 0.15 gate discriminates within) |

### Layer 2 — Hard gates (NaN-permissive)
| Gate | Threshold |
|---|---|
| `pop_yoy_pct` | ≥ 0.0% (flat OK, shrinking blocked) |
| `employment_diversity_hhi` | ≤ 0.15 |
| `permits_intensity_ratio` | < 1.5 |
| `unemployment_rate` | ≤ 6.0% |
| `wage_growth_yoy` | ≥ 2.0% |
| `max_supersector_share` | ≤ 25% |
| `cap_spread_to_gateway_pp` | ≥ 0.0 |
| `any_disqualifier` | must be FALSE |

### Layer 3 — Cap floor
Cap ≥ pool median (yield floor within qualified pool).

### Layer 4 — Opportunity score
| Signal | Weight |
|---|---|
| `cap_spread_to_gateway_pp` | **1.5** (crown, tuned from 2.0) |
| `cap_rate_proxy` | **1.5** |
| `wage_growth_yoy` | **0.8** |
| `pop_yoy_pct` | **0.5** |
| `business_apps_per_1000` | **0.5** |
| `permits_per_new_job` | **-1.0** (inverted supply-wave penalty) |
| `recession_resilience_score` | **0.5** |
| `knowledge_emp_growth_vs_state_pp` | **0.5** |
| `rent_growth_vs_wage_growth` | **0.7** (tuned from 0.3) |

### Layer 5 — State cap + top-N
Walk down score, skip if state has hit 3 picks, take top 10.

### Horizon difference vs 3yr notebook
Every performance metric — cash flow, appreciation, total return, levered ROE, stress counterfactuals, current-year underwriting — evaluates a **5-year hold** using `*_forward_return_60mo` columns and `5 × mortgage_rate` debt service, etc. The strategy method (which markets to pick) is identical.


## 1. Config

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG — EMERGING-VALUE with 5-YEAR hold horizon
# ══════════════════════════════════════════════════════════════════════════
import duckdb
import numpy as np
import pandas as pd

WAREHOUSE = '/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb'

# EMERGING-VALUE universe (broadened HHI so strategy HHI gate at 0.15 discriminates)
POP_MIN            = 300_000
POP_MAX            = 1_000_000
HHI_MAX            = 0.20
TOP_N              = 10
MAX_PER_STATE      = 3
MAX_SECTOR_SHARE   = 0.25
MIN_CAP_SPREAD_PP  = 0.0
MAX_HHI_STRATEGY   = 0.15

# ── HORIZON KNOB (this is the ONE difference from the 3yr notebook) ──
HORIZON_YEARS = 5
LTV           = 0.70

STRATEGY_NAME = 'EMERGING-VALUE (5yr hold)'
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 200)
print(f'Config loaded. Universe: pop {POP_MIN:,}-{POP_MAX:,}  HHI<={HHI_MAX}  top_n={TOP_N}  HORIZON={HORIZON_YEARS}yr')


## 2. Load universe + compute 5-year return metrics

In [ ]:
# Load universe + attach mortgage rate + opex + compute 5-YEAR return metrics
con = duckdb.connect(WAREHOUSE, read_only=True)
uni = con.execute(f'''
    SELECT fips, county_name, state_name, pop_latest,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           cap_rate_proxy, zori_yoy, mf_cap_rate_change_3yr,
           unemployment_rate, unemployment_rate_yoy_chg,
           rental_vacancy_rate, renter_share,
           hpi_yoy, hpi_trailing_24mo, permits_intensity_ratio,
           any_disqualifier,
           domesticmig_per_1000, domesticmig_accel_3yr,
           permits_per_1000,
           wage_growth_yoy, emp_growth_yoy, rent_growth_vs_wage_growth,
           business_apps_per_1000, business_apps_per_1000_3yr_avg,
           knowledge_emp_yoy, knowledge_emp_growth_vs_state_pp, renter_ind_emp_yoy,
           age_20_34_share_yoy_chg,
           max_supersector_share, employment_diversity_hhi,
           zori_forward_return_60mo, hpi_forward_return_60mo
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
      AND pop_latest > {POP_MIN} AND pop_latest <= {POP_MAX}
      AND employment_diversity_hhi IS NOT NULL
      AND employment_diversity_hhi <= {HHI_MAX}
''').df()

mortgage = con.execute('''
    SELECT EXTRACT(YEAR FROM observation_date)::INT AS year, value / 100.0 AS mortgage_rate
    FROM main_staging.stg_fred_macro
    WHERE series_id = 'MORTGAGE30US'
    QUALIFY row_number() OVER (
        PARTITION BY EXTRACT(YEAR FROM observation_date)
        ORDER BY observation_date DESC) = 1
''').df()

pep_raw = con.execute('''
    SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS ref_year,
           popestimate, as_of_date
    FROM main_staging.stg_census_pep
''').df()

# MF unit counts by structure size (PIT-strict)
mf_units_raw = con.execute('''
    SELECT fips,
           EXTRACT(YEAR FROM period_end_date)::INT AS ref_year,
           units_in_5_to_9_bldgs   AS units_5to9,
           units_in_10_to_19_bldgs AS units_10to19,
           units_in_20_to_49_bldgs AS units_20to49,
           units_in_50_plus_bldgs  AS units_50plus,
           as_of_date
    FROM main_staging.stg_census_acs_1yr
''').df()
con.close()
pep_raw['as_of_date']      = pd.to_datetime(pep_raw['as_of_date'])
mf_units_raw['as_of_date'] = pd.to_datetime(mf_units_raw['as_of_date'])

def pit_mf_units(cohort_year):
    cutoff = pd.Timestamp(f'{cohort_year}-12-31')
    df = mf_units_raw[mf_units_raw['as_of_date'] <= cutoff].copy()
    if df.empty:
        return pd.DataFrame(columns=['fips','mf_bldgs_5to9','mf_bldgs_10to19','mf_bldgs_5to19','mf_bldgs_20to49','mf_bldgs_50plus'])
    df = df.sort_values(['fips','ref_year','as_of_date'], ascending=[True, False, False])
    df = df.drop_duplicates(subset=['fips'], keep='first')
    df['mf_bldgs_5to9']    = (df['units_5to9']    / 7.0).round().astype('Int64')
    df['mf_bldgs_10to19']  = (df['units_10to19']  / 14.5).round().astype('Int64')
    df['mf_bldgs_20to49']  = (df['units_20to49']  / 34.5).round().astype('Int64')
    df['mf_bldgs_50plus']  = (df['units_50plus']  / 75.0).round().astype('Int64')
    df['mf_bldgs_5to19']   = df['mf_bldgs_5to9'] + df['mf_bldgs_10to19']
    return df[['fips','mf_bldgs_5to9','mf_bldgs_10to19','mf_bldgs_5to19','mf_bldgs_20to49','mf_bldgs_50plus']]

mf_units_by_year = []
for yr in sorted(uni['year'].unique()):
    d = pit_mf_units(yr).copy()
    d['year'] = yr
    mf_units_by_year.append(d)
mf_units_df = pd.concat(mf_units_by_year, ignore_index=True) if mf_units_by_year else pd.DataFrame()
uni = uni.merge(mf_units_df, on=['fips','year'], how='left')
uni = uni.merge(mortgage, on='year', how='left')

# Gateway cap spread
GATEWAY_FIPS = ['06037','36061','36047','36081','06075','25025','11001','17031']
con2 = duckdb.connect(WAREHOUSE, read_only=True)
gateway_caps = con2.execute(f'''
    SELECT EXTRACT(YEAR FROM period_end_date)::INT AS year,
           MEDIAN(cap_rate_proxy) AS gateway_cap_median
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
      AND fips IN ({",".join(repr(f) for f in GATEWAY_FIPS)})
    GROUP BY year ORDER BY year
''').df()
con2.close()
uni = uni.merge(gateway_caps, on='year', how='left')
uni['cap_spread_to_gateway_pp'] = (uni['cap_rate_proxy'] - uni['gateway_cap_median']) * 100
uni['hpi_rent_gap_pp']          = (uni['hpi_yoy'].fillna(0) - uni['zori_yoy'].fillna(0)) * 100
uni['permits_per_new_job']      = uni['permits_per_1000'] / (uni['emp_growth_yoy'].clip(lower=0.005) * 450)

# State-based opex ratios
OPEX_BY_STATE = {
    'Ohio':0.57,'Michigan':0.57,'Pennsylvania':0.57,'Illinois':0.57,'Wisconsin':0.57,
    'New York':0.57,'New Jersey':0.57,'Connecticut':0.57,
    'Missouri':0.52,'Massachusetts':0.52,'Maryland':0.52,'Rhode Island':0.52,
    'New Hampshire':0.52,'Vermont':0.52,'Maine':0.52,
    'District of Columbia':0.52,'Hawaii':0.52,
    'California':0.47,'Iowa':0.47,'Minnesota':0.47,'Alaska':0.47,
    'Florida':0.47,'Indiana':0.47,
    'Texas':0.44,'Virginia':0.44,'Colorado':0.44,'Oregon':0.44,'Washington':0.44,
    'Kansas':0.44,'Nebraska':0.44,'North Dakota':0.44,
    'West Virginia':0.44,'Delaware':0.44,'Kentucky':0.44,'Louisiana':0.44,
    'Georgia':0.42,'North Carolina':0.42,'South Carolina':0.42,'Tennessee':0.42,
    'Oklahoma':0.42,'Utah':0.42,'Idaho':0.42,'Montana':0.42,'New Mexico':0.42,
    'Arizona':0.40,'Nevada':0.40,'Alabama':0.40,'Mississippi':0.40,
    'South Dakota':0.40,'Wyoming':0.40,'Arkansas':0.40,
}
COASTAL_FL_FIPS = {'12086','12011','12099','12071','12021','12081','12103'}
def _opex_for(row):
    if row['fips'] in COASTAL_FL_FIPS: return 0.50
    return OPEX_BY_STATE.get(row['state_name'], 0.45)
uni['opex_ratio'] = uni.apply(_opex_for, axis=1)

# ── RECESSION RESILIENCE SCORE (RRS) — FORWARD-LOOKING (CVS methodology) ──
# Insight: markets don't have permanent crash risk based on geography. The NEXT crash
# hits markets that CURRENTLY resemble the profile of PAST crash markets.
# 5 forward-looking factors (each 0-10, higher = MORE RESILIENT):
#   Y (0.30) — Yield cushion (cap_rate_proxy)
#   B (0.30) — Bubble check (trailing 5yr HPI) — inverted
#   D (0.15) — Diversification (employment_diversity_hhi)
#   M (0.15) — Migration extreme (pop_yoy) — inverted (>4%/yr = speculative)
#   P (0.10) — Permit intensity — inverted (>1.5 = supply wave)
_con_rrs = duckdb.connect(WAREHOUSE, read_only=True)
_all_fips_str = ",".join(repr(f) for f in uni['fips'].unique().tolist())
_hpi_all = _con_rrs.execute(f'''
    SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS year, hpi
    FROM main_staging.stg_fhfa_hpi
    WHERE fips IN ({_all_fips_str}) AND EXTRACT(MONTH FROM period_end_date) = 12
    QUALIFY row_number() OVER (PARTITION BY fips, EXTRACT(YEAR FROM period_end_date) ORDER BY as_of_date DESC) = 1
''').df()
_con_rrs.close()
_lag5 = _hpi_all.rename(columns={'hpi':'hpi_5yr_prior','year':'year_join'})
_lag5['year_join'] = _lag5['year_join'] + 5
_hpi_all = _hpi_all.merge(_lag5[['fips','year_join','hpi_5yr_prior']], left_on=['fips','year'], right_on=['fips','year_join'], how='left').drop(columns='year_join')
_hpi_all['trailing_5yr_hpi_ann'] = (_hpi_all['hpi'] / _hpi_all['hpi_5yr_prior']) ** (1/5) - 1
uni = uni.merge(_hpi_all[['fips','year','trailing_5yr_hpi_ann']], on=['fips','year'], how='left')

def _rrs_yield(v):
    if pd.isna(v): return 5
    p = v * 100
    if p >= 8.0: return 10
    if p >= 7.0: return 8
    if p >= 6.5: return 6
    if p >= 6.0: return 4
    if p >= 5.5: return 2
    return 0
def _rrs_bubble_inv(v):
    '''Trailing 5yr HPI — INVERTED (extreme growth = risky)'''
    if pd.isna(v): return 5
    p = v * 100
    if p < 3: return 10
    if p < 5: return 8
    if p < 8: return 6
    if p < 10: return 4
    if p < 12: return 2
    return 0
def _rrs_hhi(v):
    if pd.isna(v): return 5
    if v < 0.06: return 10
    if v < 0.08: return 8
    if v < 0.10: return 6
    if v < 0.12: return 4
    if v < 0.15: return 2
    return 0
def _rrs_permit_inv(v):
    '''Permit intensity — INVERTED (supply wave = risky)'''
    if pd.isna(v): return 5
    if v <= 0.8: return 10
    if v <= 1.0: return 8
    if v <= 1.2: return 6
    if v <= 1.5: return 4
    if v <= 2.0: return 2
    return 0

uni['rrs_Y'] = uni['cap_rate_proxy'].apply(_rrs_yield)
uni['rrs_B'] = uni['trailing_5yr_hpi_ann'].apply(_rrs_bubble_inv)
uni['rrs_D'] = uni['employment_diversity_hhi'].apply(_rrs_hhi)
uni['rrs_P'] = uni['permits_intensity_ratio'].apply(_rrs_permit_inv)
# rrs_M (migration) and final recession_resilience_score computed after pop_yoy_cache is built

# ── 5-YEAR RETURN METRICS ──────────────────────────────────────────────────
uni['cash_flow_5yr']    = HORIZON_YEARS * uni['cap_rate_proxy'] * (1 - uni['opex_ratio']) * (1 + uni['zori_forward_return_60mo'] / 2)
uni['appreciation_5yr'] = uni['hpi_forward_return_60mo']
uni['total_return_5yr'] = uni['cash_flow_5yr'] + uni['appreciation_5yr']
uni['debt_service_5yr'] = HORIZON_YEARS * LTV * uni['mortgage_rate']
uni['levered_ROE_5yr']  = (uni['cash_flow_5yr'] - uni['debt_service_5yr'] + uni['appreciation_5yr']) / (1 - LTV)

def pit_pop_yoy(cohort_year):
    cutoff = pd.Timestamp(f'{cohort_year}-12-31')
    df = pep_raw[pep_raw['as_of_date'] <= cutoff].copy()
    if df.empty: return pd.DataFrame(columns=['fips','pop_yoy_pct'])
    df = df.sort_values(['fips','ref_year','as_of_date']).drop_duplicates(subset=['fips','ref_year'], keep='last')
    rows=[]
    for f, sub in df.groupby('fips'):
        sub = sub.sort_values('ref_year', ascending=False)
        if len(sub) < 2: continue
        latest, prev = sub.iloc[0], sub[sub['ref_year']==sub.iloc[0]['ref_year']-1]
        if prev.empty: continue
        prev = prev.iloc[0]
        rows.append({'fips':f,'pop_yoy_pct':(latest['popestimate']-prev['popestimate'])/prev['popestimate']*100})
    return pd.DataFrame(rows)
pop_yoy_cache = {yr: pit_pop_yoy(yr) for yr in range(2015, 2026)}

# ── Complete RRS: attach pop_yoy per (fips, year), compute migration + final score ──
_pop_yoy_all = pd.concat([df.assign(year=yr) for yr, df in pop_yoy_cache.items() if not df.empty], ignore_index=True) if any(not df.empty for df in pop_yoy_cache.values()) else pd.DataFrame()
if not _pop_yoy_all.empty:
    uni = uni.merge(_pop_yoy_all[['fips','year','pop_yoy_pct']].rename(columns={'pop_yoy_pct':'pop_yoy_pct_rrs'}), on=['fips','year'], how='left')
else:
    uni['pop_yoy_pct_rrs'] = float('nan')

def _rrs_migration_inv(v):
    '''Pop growth — INVERTED (extreme migration = speculative = risky)'''
    if pd.isna(v): return 5
    if v < 0: return 4
    if v <= 1: return 10
    if v <= 2: return 8
    if v <= 3: return 6
    if v <= 4: return 4
    return 2
uni['rrs_M'] = uni['pop_yoy_pct_rrs'].apply(_rrs_migration_inv)
uni['recession_resilience_score'] = (0.30*uni['rrs_Y'] + 0.30*uni['rrs_B'] + 0.15*uni['rrs_D'] +
                                      0.15*uni['rrs_M'] + 0.10*uni['rrs_P'])

n_obs = uni['total_return_5yr'].notna().sum()
print(f'Universe: {len(uni)} county-years across {uni.year.nunique()} years ({uni.year.min()}-{uni.year.max()})')
print(f'5yr forward observable: {n_obs} county-years (2015-2020 cohorts)')


## 3. Method implementation

In [ ]:
# EMERGING-VALUE — gates + weights UNCHANGED from 3yr notebook
EMERGING_VALUE_GATES = {
    'min_pop_yoy_pct':      0.0,
    'max_permit_ratio':     1.5,
    'max_unemp_pct':        6.0,
    'min_wage_growth':      0.02,
    'max_sector_share':     MAX_SECTOR_SHARE,
    'min_cap_spread_pp':    MIN_CAP_SPREAD_PP,
    'max_hhi_strategy':     MAX_HHI_STRATEGY,
    'require_no_disq':      True,
}
OPPORTUNITY_WEIGHTS = {
    'cap_spread_to_gateway_pp':          1.5,   # tuned from 2.0 via 100K-500K walk-forward
    'cap_rate_proxy':                    1.5,
    'wage_growth_yoy':                   0.8,
    'pop_yoy_pct':                       0.5,
    'business_apps_per_1000':            0.5,
    'permits_per_new_job':              -1.0,
    'recession_resilience_score':        0.5,   # forward-looking recession vulnerability
    'knowledge_emp_growth_vs_state_pp':  0.5,   # premium-worker growth vs state (Tier 1a validated)
    'rent_growth_vs_wage_growth':        0.7,   # rent capacity gap (tuned from 0.3 via walk-forward)
}

def pick_for_year(year, gates=EMERGING_VALUE_GATES):
    c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()
    if c.empty: return pd.DataFrame()
    pop_yr = pop_yoy_cache.get(year, pd.DataFrame())
    if not pop_yr.empty:
        c = c.merge(pop_yr[['fips','pop_yoy_pct']], on='fips', how='left')
    else:
        c['pop_yoy_pct'] = np.nan
    n_before = len(c)

    c = c[c['pop_yoy_pct'].isna()             | (c['pop_yoy_pct']             >= gates['min_pop_yoy_pct'])]
    c = c[c['permits_intensity_ratio'].isna() | (c['permits_intensity_ratio'] <  gates['max_permit_ratio'])]
    c = c[c['unemployment_rate'].isna()       | (c['unemployment_rate']       <= gates['max_unemp_pct'])]
    c = c[c['wage_growth_yoy'].isna()         | (c['wage_growth_yoy']         >= gates['min_wage_growth'])]
    c = c[c['max_supersector_share'].isna()   | (c['max_supersector_share']   <= gates['max_sector_share'])]
    c = c[c['cap_spread_to_gateway_pp'].isna()| (c['cap_spread_to_gateway_pp']>= gates['min_cap_spread_pp'])]
    c = c[c['employment_diversity_hhi'].isna()| (c['employment_diversity_hhi']<= gates['max_hhi_strategy'])]
    if gates['require_no_disq']:
        c = c[(c['any_disqualifier'] == False) | (c['any_disqualifier'].isna())]
    if len(c) > 0:
        cap_median = c['cap_rate_proxy'].median()
        c = c[c['cap_rate_proxy'] >= cap_median]
    n_after = len(c)
    print(f'  {year}: {n_before} → {n_after} candidates after value gates ({n_after/n_before*100:.0f}% pass)')
    if c.empty: return pd.DataFrame()

    # Cap business_apps_per_1000 at 40 to prevent incorporation-haven artifacts
    # (Delaware, Wyoming show 50-120+ due to corporate registrations)
    c = c.copy()
    if 'business_apps_per_1000' in c.columns:
        c['business_apps_per_1000'] = c['business_apps_per_1000'].clip(upper=40)
    c['opportunity_score'] = 0.0
    for col, w in OPPORTUNITY_WEIGHTS.items():
        if col in c.columns and c[col].notna().any():
            mu, sig = c[col].mean(), c[col].std()
            if sig > 0:
                z = (c[col] - mu) / sig
                c['opportunity_score'] = c['opportunity_score'] + w * z.fillna(0)
    c = c.sort_values('opportunity_score', ascending=False)

    picked, sc = [], {}
    for _, r in c.iterrows():
        if sc.get(r['state_name'], 0) >= MAX_PER_STATE: continue
        picked.append(r); sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
        if len(picked) >= TOP_N: break
    return pd.DataFrame(picked) if picked else pd.DataFrame()

sample = pick_for_year(2025)
if not sample.empty:
    print(f'\nEMERGING-VALUE 5yr picks for 2025:')
    print(sample[['county_name','state_name','pop_latest','cap_rate_proxy','cap_spread_to_gateway_pp','pop_yoy_pct','wage_growth_yoy','recession_resilience_score','trailing_5yr_hpi_ann','opportunity_score']].to_string(index=False))


## 4. Per-year picks (2015–2025)

In [ ]:
# Per-year picks with 5yr realized performance
def _name(row): return f"{row['county_name'].replace(' County','')}, {row['state_name']}"

def _print_row(i, row, obs=True):
    n   = _name(row)
    cap = row['cap_rate_proxy'] * 100
    opx = row['opex_ratio']     * 100
    mtg = row['mortgage_rate']  * 100
    if obs and pd.notna(row.get('total_return_5yr')):
        cf  = row['cash_flow_5yr']    * 100
        ap  = row['appreciation_5yr'] * 100
        tr  = row['total_return_5yr'] * 100
        roe = row['levered_ROE_5yr']  * 100
        tr_ann = ((1+tr/100)**(1/5)-1)*100 if 1+tr/100 > 0 else float('nan')
        print(f'  #{i+1:<2}  {n:<32}  cap {cap:>5.2f}%  opex {opx:>3.0f}%  mtg {mtg:>4.2f}%   cf_net {cf:>+6.2f}%  apprec {ap:>+7.2f}%   TOTAL {tr:>+7.2f}% ({tr_ann:>+5.2f}%/yr)   LEV ROE (70%LTV) {roe:>+7.2f}%')
    else:
        net_yield = cap * (1 - row['opex_ratio'])
        spread = net_yield - mtg
        zy = row['zori_yoy'] * 100 if pd.notna(row.get('zori_yoy')) else float('nan')
        print(f'  #{i+1:<2}  {n:<32}  cap {cap:>5.2f}%  opex {opx:>3.0f}%  net_yield {net_yield:>4.2f}%  mtg {mtg:>4.2f}%  spread {spread:>+5.2f}pp  zori_yoy {zy:>+5.2f}%   (60mo n/a)')

years = sorted(uni['year'].unique().tolist())
for yr in years:
    if yr < 2015: continue
    picks = pick_for_year(yr)
    if picks is None or picks.empty: continue
    obs = pd.notna(picks['total_return_5yr']).any()
    yr_uni = uni[uni['year']==yr].dropna(subset=['total_return_5yr'])
    print('\n' + '=' * 140)
    if obs and not yr_uni.empty:
        u_mean = yr_uni['total_return_5yr'].mean() * 100
        u_med  = yr_uni['total_return_5yr'].median() * 100
        print(f'YEAR {yr}   universe_n={len(uni[uni.year==yr])}   universe 5yr TR: mean {u_mean:+.2f}%   median {u_med:+.2f}%')
    else:
        print(f'YEAR {yr}   universe_n={len(uni[uni.year==yr])}   (5yr forward not observable yet — ex-ante view)')
    print('=' * 140)
    for i in range(len(picks)):
        _print_row(i, picks.iloc[i], obs=obs)
    if obs and picks['total_return_5yr'].notna().any():
        cf_avg  = picks['cash_flow_5yr'].mean()    * 100
        ap_avg  = picks['appreciation_5yr'].mean() * 100
        tr_avg  = picks['total_return_5yr'].mean() * 100
        roe_avg = picks['levered_ROE_5yr'].mean()  * 100
        opx_avg = picks['opex_ratio'].mean() * 100
        edge    = tr_avg - u_mean
        print(f'       {"AVG":<32}                        opex {opx_avg:>3.0f}%              cf_net {cf_avg:>+6.2f}%  apprec {ap_avg:>+7.2f}%   TOTAL {tr_avg:>+7.2f}%   LEV ROE {roe_avg:>+7.2f}%   [edge {edge:+.2f}pp]')


## 5. Aggregate performance (2015–2020 observable)

In [ ]:
# Aggregate performance (2015-2020 observable cohorts — 5yr forward)
rows = []
for yr in years:
    if yr < 2015: continue
    picks = pick_for_year(yr)
    if picks is None or picks.empty or picks['total_return_5yr'].isna().all(): continue
    yr_uni = uni[uni['year'] == yr].dropna(subset=['total_return_5yr'])
    rows.append({
        'year':              yr,
        'n_picks':           len(picks),
        'cf_net':            picks['cash_flow_5yr'].mean() * 100,
        'apprec':            picks['appreciation_5yr'].mean() * 100,
        'total_return':      picks['total_return_5yr'].mean() * 100,
        'levered_ROE_70LTV': picks['levered_ROE_5yr'].mean() * 100,
        'universe_mean':     yr_uni['total_return_5yr'].mean() * 100,
        'edge_pp':           (picks['total_return_5yr'].mean() - yr_uni['total_return_5yr'].mean()) * 100,
    })
agg = pd.DataFrame(rows)
print('Year-by-year (2015-2020 observable, 5yr forward):')
print(agg.to_string(index=False))
if not agg.empty:
    print()
    print(f'  Avg NET cash flow (5yr):     {agg["cf_net"].mean():>+7.2f}%')
    print(f'  Avg appreciation (5yr):      {agg["apprec"].mean():>+7.2f}%')
    print(f'  Avg TOTAL return (5yr, net): {agg["total_return"].mean():>+7.2f}%')
    print(f'  Avg LEV ROE 70% LTV (5yr):   {agg["levered_ROE_70LTV"].mean():>+7.2f}%')
    print(f'  Avg edge vs universe:        {agg["edge_pp"].mean():>+7.2f}pp')


## 6. Per-pick history — every pick with realized 5yr returns

### Per-pick history — every county this strategy picked, and how it actually performed

One row per pick across all cohort years. Shows what the strategy chose (the "prediction") and
what the market actually delivered over the 3-year hold (the "actual result").

**Verdict = relative performance vs the universe average** (NOT whether the pick made money):
- 🟢 GREEN = **ABOVE AVERAGE** — pick beat the universe average for that year by > 2pp
- 🟡 AMBER = **AVERAGE** — pick was within ±2pp of universe average
- ⚪ GREY  = **BELOW AVERAGE** — pick trailed the universe by > 2pp (still often profitable!)
- ⚪ GRAY  = **n/a** — 3yr forward return not yet observable (2023+ cohorts)

**Separate red highlighting** on `total_return_3yr%` and `levered_ROE_3yr%` when the ABSOLUTE return is NEGATIVE (pick actually lost money in dollars — a true loss, not just underperformance).

Important distinction: a pick that returned +30% in a +45% cohort is **BELOW AVERAGE** but still made money.
Only picks with negative absolute returns get the red highlight.

**How to read**: sort by year to see each cohort's picks, sort by `total_return_3yr%` to see best/worst picks overall,
filter `verdict` == 'ABOVE AVERAGE' to see which picks materially beat the market, etc.


In [ ]:
# Per-pick history — every pick, every cohort, with realized 5-YEAR performance,
# NOI vs cap-compression decomposition, and counterfactual stress-test ROEs.
picks_rows = []
for yr in years:
    if yr < 2015: continue
    picks = pick_for_year(yr)
    if picks is None or picks.empty: continue
    yr_uni = uni[uni['year'] == yr].dropna(subset=['total_return_5yr'])
    uni_mean_tr = yr_uni['total_return_5yr'].mean() * 100 if not yr_uni.empty else float('nan')
    for i in range(len(picks)):
        row = picks.iloc[i]
        cf   = row['cash_flow_5yr']    * 100 if pd.notna(row.get('cash_flow_5yr'))    else None
        ap   = row['appreciation_5yr'] * 100 if pd.notna(row.get('appreciation_5yr')) else None
        tr   = row['total_return_5yr'] * 100 if pd.notna(row.get('total_return_5yr')) else None
        roe  = row['levered_ROE_5yr']  * 100 if pd.notna(row.get('levered_ROE_5yr'))  else None
        cap  = row['cap_rate_proxy']   * 100
        opex = row['opex_ratio']       * 100
        mtg  = row['mortgage_rate']    * 100
        edge = (tr - uni_mean_tr) if (tr is not None and pd.notna(uni_mean_tr)) else None
        tr_ann = ((1+tr/100)**(1/5)-1)*100 if (tr is not None and 1+tr/100 > 0) else None

        # NOI vs Cap Compression Decomposition (using 5yr ZORI + HPI)
        zori_60mo = row.get('zori_forward_return_60mo')
        hpi_60mo  = row.get('hpi_forward_return_60mo')
        entry_noi_gross = cap * (1 - opex/100)
        if pd.notna(zori_60mo):
            exit_noi_gross = entry_noi_gross * (1 + zori_60mo)
            noi_growth_pct = zori_60mo * 100
        else:
            exit_noi_gross = None; noi_growth_pct = None
        if pd.notna(hpi_60mo) and pd.notna(zori_60mo):
            exit_cap_gross = cap * (1 + zori_60mo) / (1 + hpi_60mo)
            cap_change_pp  = cap - exit_cap_gross
            apprec_from_noi = zori_60mo * 100
            apprec_from_cap = (hpi_60mo - zori_60mo) * 100
        else:
            exit_cap_gross = None; cap_change_pp = None
            apprec_from_noi = None; apprec_from_cap = None

        # ── COUNTERFACTUAL STRESS TESTS (5yr versions) ─────────────────────
        # A: mortgage +100bp for 5 years  |  B: NOI cap +50bp at exit  |  Both combined
        HORIZON = 5
        LTV     = 0.70
        ROE_stress_rate = ROE_stress_cap = ROE_stress_both = None
        if pd.notna(cf) and pd.notna(zori_60mo):
            cf_frac = cf / 100
            actual_debt_frac = HORIZON * LTV * (mtg / 100)
            actual_apprec_frac = (ap / 100) if pd.notna(ap) else 0.0
            # Stress A: rate +100bp for full 5yr hold
            stressed_debt_frac = HORIZON * LTV * ((mtg + 1.0) / 100)
            ROE_stress_rate = (cf_frac - stressed_debt_frac + actual_apprec_frac) / (1 - LTV) * 100
            # Stress B: NOI cap +50bp at exit
            entry_noi_cap  = cap/100 * (1 - opex/100)
            exit_noi       = entry_noi_cap * (1 + zori_60mo)
            stressed_exit_cap = entry_noi_cap + 0.005
            stressed_exit_val = exit_noi / stressed_exit_cap
            stressed_apprec_frac = stressed_exit_val - 1
            ROE_stress_cap = (cf_frac - actual_debt_frac + stressed_apprec_frac) / (1 - LTV) * 100
            # Both combined
            ROE_stress_both = (cf_frac - stressed_debt_frac + stressed_apprec_frac) / (1 - LTV) * 100

        if tr is None:
            verdict = 'n/a — not yet observable'
        elif edge > 2:
            verdict = 'ABOVE AVERAGE'
        elif edge < -2:
            verdict = 'BELOW AVERAGE'
        else:
            verdict = 'AVERAGE'

        picks_rows.append({
            'year':                 yr,
            'rank':                 i+1,
            'market':               f"{row['county_name'].replace(' County','')}, {row['state_name']}",
            'entry_cap%':           cap,
            'exit_cap%':            exit_cap_gross,
            'cap_comp_pp':          cap_change_pp,
            'opex%':                opex,
            'entry_noi%':           entry_noi_gross,
            'exit_noi%':            exit_noi_gross,
            'noi_growth_5yr%':      noi_growth_pct,
            'mtg%':                 mtg,
            'cf_net_5yr%':          cf,
            'apprec_from_noi%':     apprec_from_noi,
            'apprec_from_cap%':     apprec_from_cap,
            'apprec_total_5yr%':    ap,
            'total_return_5yr%':    tr,
            'total_return_ann%':    tr_ann,
            'ROE_stress_rate%':     ROE_stress_rate,
            'ROE_stress_cap%':      ROE_stress_cap,
            'ROE_stress_both%':     ROE_stress_both,
            'universe_TR_5yr%':     uni_mean_tr if pd.notna(uni_mean_tr) else None,
            'edge_pp':              edge,
            'levered_ROE_5yr%':     roe,
            'verdict':              verdict,
            'mf_bldgs_5to9':        int(row['mf_bldgs_5to9'])   if pd.notna(row.get('mf_bldgs_5to9'))   else None,
            'mf_bldgs_10to19':      int(row['mf_bldgs_10to19']) if pd.notna(row.get('mf_bldgs_10to19')) else None,
            'mf_bldgs_5to19':       int(row['mf_bldgs_5to19'])  if pd.notna(row.get('mf_bldgs_5to19'))  else None,
            'mf_bldgs_20to49':      int(row['mf_bldgs_20to49']) if pd.notna(row.get('mf_bldgs_20to49')) else None,
            'mf_bldgs_50plus':      int(row['mf_bldgs_50plus']) if pd.notna(row.get('mf_bldgs_50plus')) else None,
        })
picks_df = pd.DataFrame(picks_rows)

def _color_verdict_pick(v):
    if v == 'ABOVE AVERAGE':  return 'background-color: #d4edda; color: #006600; font-weight: bold'
    if v == 'BELOW AVERAGE':  return 'background-color: #e2e3e5; color: #6c757d'
    if v == 'AVERAGE':        return 'background-color: #fff3cd; color: #7a5a00'
    return 'background-color: #eeeeee; color: #666'
def _color_edge(v):
    if pd.isna(v): return ''
    if v > 5:  return 'color: #006600; font-weight: bold'
    if v > 0:  return 'color: #006600'
    if v < -5: return 'color: #b30000; font-weight: bold'
    return 'color: #b30000'
def _color_negative_return(v):
    if pd.isna(v): return ''
    if v < 0: return 'background-color: #f8d7da; color: #b30000; font-weight: bold'
    return ''

picks_styled = (picks_df.style
    .format({
        'entry_cap%':'{:.2f}%', 'exit_cap%':'{:.2f}%', 'cap_comp_pp':'{:+.2f}pp',
        'opex%':'{:.0f}%', 'entry_noi%':'{:.2f}%', 'exit_noi%':'{:.2f}%',
        'noi_growth_5yr%':'{:+.2f}%', 'mtg%':'{:.2f}%',
        'cf_net_5yr%':'{:+.2f}%',
        'apprec_from_noi%':'{:+.2f}%', 'apprec_from_cap%':'{:+.2f}%', 'apprec_total_5yr%':'{:+.2f}%',
        'total_return_5yr%':'{:+.2f}%', 'total_return_ann%':'{:+.2f}%/yr',
        'universe_TR_5yr%':'{:+.2f}%', 'edge_pp':'{:+.2f}pp',
        'levered_ROE_5yr%':'{:+.2f}%',
        'ROE_stress_rate%':'{:+.2f}%', 'ROE_stress_cap%':'{:+.2f}%', 'ROE_stress_both%':'{:+.2f}%',
        'mf_bldgs_5to9':'{:,.0f}', 'mf_bldgs_10to19':'{:,.0f}', 'mf_bldgs_5to19':'{:,.0f}',
        'mf_bldgs_20to49':'{:,.0f}', 'mf_bldgs_50plus':'{:,.0f}',
    }, na_rep='—')
    .map(_color_verdict_pick, subset=['verdict'])
    .map(_color_edge, subset=['edge_pp'])
    .map(_color_negative_return, subset=['total_return_5yr%'])
    .map(_color_negative_return, subset=['levered_ROE_5yr%'])
    .map(_color_negative_return, subset=['ROE_stress_rate%'])
    .map(_color_negative_return, subset=['ROE_stress_cap%'])
    .map(_color_negative_return, subset=['ROE_stress_both%'])
    .set_caption('Every pick across every cohort — 5yr hold. Red = absolute loss. Stress columns = counterfactual 5yr ROE at +100bp mortgage / +50bp exit NOI cap / both.')
)
picks_styled


In [ ]:
# Aggregate: how did this strategy's picks perform historically? (5yr hold)
observable = picks_df[picks_df['verdict'].isin(['ABOVE AVERAGE','BELOW AVERAGE','AVERAGE'])].copy()
n_above   = int((observable['verdict']=='ABOVE AVERAGE').sum())
n_avg     = int((observable['verdict']=='AVERAGE').sum())
n_below   = int((observable['verdict']=='BELOW AVERAGE').sum())
n_underwater = int((observable['total_return_5yr%'] < 0).sum())
n_total   = len(observable)

STRATEGY_NAME = globals().get('STRATEGY_NAME', 'EMERGING-VALUE (5yr hold)')

summary_rows = [
    {'metric': f'Strategy',                                    'result': STRATEGY_NAME},
    {'metric': f'Above average (beat universe by >2pp)',       'result': f'{n_above}/{n_total}'},
    {'metric': f'Average (within ±2pp of universe)',           'result': f'{n_avg}/{n_total}'},
    {'metric': f'Below average (trailed universe by >2pp)',    'result': f'{n_below}/{n_total}'},
    {'metric': f'Underwater deals (negative total return)',    'result': f'{n_underwater}/{n_total}'},
]
print('STRATEGY SUMMARY:')
print()
print(pd.DataFrame(summary_rows).to_string(index=False))
print()

# Stress-test survival summary
n_stress_rate_neg = int((observable['ROE_stress_rate%'] < 0).sum())
n_stress_cap_neg  = int((observable['ROE_stress_cap%'] < 0).sum())
n_stress_both_neg = int((observable['ROE_stress_both%'] < 0).sum())
detail_rows = [
    {'metric': 'above-average rate %',                                     'value': f'{round(n_above/n_total*100, 1)}%' if n_total else '—'},
    {'metric': 'avg edge vs universe (pp)',                                'value': f"{observable['edge_pp'].mean():+.2f}pp"},
    {'metric': 'avg total return (5yr)',                                   'value': f"{observable['total_return_5yr%'].mean():+.2f}%"},
    {'metric': 'avg total return (annualized)',                            'value': f"{observable['total_return_ann%'].mean():+.2f}%/yr"},
    {'metric': 'avg levered ROE (70% LTV, 5yr)',                           'value': f"{observable['levered_ROE_5yr%'].mean():+.2f}%"},
    {'metric': 'best pick TR',                                             'value': f"{observable['total_return_5yr%'].max():+.2f}%"},
    {'metric': 'worst pick TR',                                            'value': f"{observable['total_return_5yr%'].min():+.2f}%"},
    {'metric': 'STRESS — rate +100bp: picks with negative ROE',            'value': f'{n_stress_rate_neg}/{n_total}'},
    {'metric': 'STRESS — cap +50bp exit: picks with negative ROE',         'value': f'{n_stress_cap_neg}/{n_total}'},
    {'metric': 'STRESS — both combined: picks with negative ROE',          'value': f'{n_stress_both_neg}/{n_total}'},
    {'metric': 'STRESS — both combined: avg ROE',                          'value': f"{observable['ROE_stress_both%'].mean():+.2f}%"},
]
print('DETAILED STATS + STRESS-TEST SURVIVAL:')
print()
print(pd.DataFrame(detail_rows).to_string(index=False))
print()

best  = observable.nlargest(5, 'total_return_5yr%')[['year','rank','market','entry_cap%','total_return_5yr%','total_return_ann%','edge_pp','verdict']]
worst = observable.nsmallest(5, 'total_return_5yr%')[['year','rank','market','entry_cap%','total_return_5yr%','total_return_ann%','edge_pp','verdict']]
print('TOP 5 PICKS by realized 5yr total return:')
print(best.to_string(index=False))
print()
print('BOTTOM 5 PICKS by realized 5yr total return:')
print(worst.to_string(index=False))


## 7. Regime context (informational — does not drive picks)

### How the regime is classified

The detector reads 5 macro signals and casts a vote for each in one of three directions:
**HOT vote** (cycle overheating), **COLD vote** (cycle stress / contraction), or NEUTRAL.

### The 5 signals + vote rules

| Signal | Source | HOT vote when | COLD vote when |
|---|---|---|---|
| **National HPI YoY** | FHFA (median across universe) | z > +1.5 (prices unusually hot) | z < −1.5 (prices unusually cold) |
| **Unemployment YoY change** | FRED UNRATE | z < −1.5 (labor tight, unemp falling fast) | z > +1.5 (unemp rising fast, recession stress) |
| **CPI YoY (inflation)** | FRED CPIAUCSL | z > +1.5 (high inflation, late cycle) | z < −1.5 (deflation, recession sign) |
| **Real Fed funds** | FRED (fed_funds − CPI) | z > +1.5 (tight money, restrictive) | z < −1.5 (very easy money) |
| **Yield curve 10y-2y** | FRED T10Y2Y | inverted (< 0) — recession lead | (no cold rule for curve) |

All z-scores use **expanding-window PIT-strict computation** (only historical data up to cohort date).
Minimum 10 prior observations required before a z-score fires (avoids noise from thin history).

### Regime state — assigned by vote consensus

Requires **2+ signals to agree** before firing HOT or COLD (prevents single-signal false positives).
HOT and COLD counts are tallied independently — the classifier checks HOT threshold first,
then COLD threshold, then defaults to MID CYCLE.

| Regime state | Qualifying votes | Implicit forward prediction | Investor read |
|---|---|---|---|
| **CYCLE TOP** | `h ≥ 3` (any 3+ HOT signals fire) | Below-trend HPI (< 3.5%/yr, ~1pp under trend) | Multiple signals extreme — bubble peaking, expect mean reversion (bearish for new buys) |
| **LATE CYCLE** | `h == 2` (exactly 2 HOT signals fire, `h < 3`) | Below-trend HPI (< 3.5%/yr) | Moderate overheating — elevated risk, cycle top approaching |
| **MID CYCLE** | `h < 2` AND `c < 2` (neither side has consensus) | Trend HPI (4.5%/yr ±2pp) | No anomaly — trust trailing signals at face value |
| **CONTRACTION** | `c == 2` (exactly 2 COLD signals fire, `c < 3`) | Above-trend HPI (> 5.5%/yr) | Stress showing — recession in progress or starting |
| **CYCLE BOTTOM** | `c ≥ 3` (any 3+ COLD signals fire) | Above-trend HPI (> 5.5%/yr, mean-reversion snapback) | Deep stress / capitulation — potentially historical buying opportunity |

**Cycle spectrum** — LATE CYCLE / CYCLE TOP sit on the *overheating* end (bearish forward);
CONTRACTION / CYCLE BOTTOM sit on the *stress* end (bullish forward, mean-reversion logic).
Intensity within each side (2 votes vs 3+) is a confidence dial, not a direction change:

```
CYCLE BOTTOM  ←  CONTRACTION  ←  MID CYCLE  →  LATE CYCLE  →  CYCLE TOP
   (3+ COLD)      (2 COLD)       (default)      (2 HOT)        (3+ HOT)
   ← bullish forward ─                           ─ bearish forward →
```

### Example qualifying-vote patterns

The 5 signals: HPI, unemployment change, CPI, real Fed funds, yield curve inversion.
Each casts one vote (HOT / COLD / NEUTRAL). Curve has no COLD rule — inversion is HOT-only.

| Year (illustrative) | Fired votes | h / c | Label |
|---|---|---|---|
| 2001 dot-com bust | HPI− (z=−1.6), Unemp+ (z=+2.0), CPI− (z=−1.6) | 0 / 3 | **CYCLE BOTTOM** |
| 2006 housing top | HPI+ (z=+1.9), RealFF+ (z=+1.7), curve inverted | 3 / 0 | **CYCLE TOP** |
| 2009 GFC depths | HPI− (z=−3.1), Unemp+ (z=+2.8), CPI− (z=−2.4), RealFF− (z=−1.9) | 0 / 4 | **CYCLE BOTTOM** |
| 2019 pre-COVID | curve inverted, Unemp− (z=−1.7) | 2 / 0 | **LATE CYCLE** |
| 2021 rent bubble | HPI+ (z=+2.3), Unemp− (z=−1.9), CPI+ (z=+1.6), curve normal | 3 / 0 | **CYCLE TOP** |
| 2011 Euro crisis | HPI− (z=−1.7), Unemp+ (z=+1.6), RealFF− (z=−1.8) | 0 / 3 | **CYCLE BOTTOM** |
| 2016 mid-expansion | none fire (all \|z\| < 1.5) | 0 / 0 | **MID CYCLE** |
| 2018 rate hikes | RealFF+ (z=+1.7), curve flattening but not inverted | 1 / 0 | **MID CYCLE** (single vote — under threshold) |

Note the 2018 case: a single HOT vote is **not enough** to fire LATE CYCLE — the 2-vote minimum
is the noise filter. This is why the detector missed COVID 2020 (single-signal unemployment shock).

Vote counts are surfaced in the validation table's `hot_votes` / `cold_votes` columns —
wide vote margins on correct calls indicate detector confidence; single-vote regimes get
folded into MID CYCLE by design.

### Historical calibration (this exact detector, on known events)

| Year | Detected regime | Actual event |
|---|---|---|
| 2001 | CYCLE BOTTOM ✓ | dot-com bust |
| 2005-2006 | LATE CYCLE ✓ | SFR bubble peaking |
| 2008 | CYCLE BOTTOM ✓ | GFC crash |
| 2009-2011 | CYCLE BOTTOM / CONTRACTION ✓ | crash + recovery |
| 2015-2020 | MID CYCLE ✓ | expansion (no false positives) |
| 2021 | CYCLE TOP ✓ | rent + HPI bubble |
| 2022 | CYCLE TOP ✓ | rate shock (TOPPING) |
| 2023-2025 | MID CYCLE ✓ | rate normalization |

Detector correctly identified all major regime events since 1990 (~90% accuracy).
Missed: 2020 COVID (single-signal shock — consensus voting requires 2+ signals).


In [ ]:
# Regime context (informational — does NOT drive picks)
# 5-signal consensus detector: HPI YoY, unemployment YoY change, CPI YoY, real Fed funds, yield curve.
# Uses expanding-window z-scores (PIT-strict). Requires 2+ signals to agree before firing HOT or COLD.

# ── Pull long-history signals ──
_con = duckdb.connect(WAREHOUSE, read_only=True)
_hpi_nat = _con.execute('''
    WITH hpi_dec AS (
        SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS year, hpi
        FROM main_staging.stg_fhfa_hpi
        QUALIFY row_number() OVER (PARTITION BY fips, EXTRACT(YEAR FROM period_end_date) ORDER BY as_of_date DESC)=1
    ),
    hpi_yoy_nat AS (
        SELECT curr.fips, curr.year, curr.hpi/prev.hpi - 1 AS hpi_yoy
        FROM hpi_dec curr JOIN hpi_dec prev ON prev.fips=curr.fips AND prev.year=curr.year-1 WHERE prev.hpi>0
    )
    SELECT year, MEDIAN(hpi_yoy) AS hpi_yoy FROM hpi_yoy_nat GROUP BY year ORDER BY year
''').df()
_fred = _con.execute('''
    SELECT year,
        MAX(CASE WHEN series_id='UNRATE' THEN value END) AS unemp,
        MAX(CASE WHEN series_id='CPIAUCSL' THEN value END) AS cpi_lvl,
        MAX(CASE WHEN series_id='FEDFUNDS' THEN value END) AS fed_funds,
        MAX(CASE WHEN series_id='T10Y2Y' THEN value END) AS yield_curve
    FROM (
        SELECT series_id, EXTRACT(YEAR FROM observation_date)::INT AS year, value,
               row_number() OVER (PARTITION BY series_id, EXTRACT(YEAR FROM observation_date) ORDER BY observation_date DESC) AS rn
        FROM main_staging.stg_fred_macro
        WHERE series_id IN ('UNRATE','CPIAUCSL','FEDFUNDS','T10Y2Y')
    ) WHERE rn=1 GROUP BY year ORDER BY year
''').df()
_con.close()

_regime = _hpi_nat.merge(_fred, on='year', how='inner').sort_values('year').reset_index(drop=True)
_regime['cpi_yoy']         = _regime['cpi_lvl'].pct_change() * 100
_regime['unemp_yoy_chg']   = _regime['unemp'].diff()
_regime['real_fed_funds']  = _regime['fed_funds'] - _regime['cpi_yoy']
_regime['curve_inverted']  = (_regime['yield_curve'] < 0).astype(int)

def _expand_z(series, min_periods=10):
    out = pd.Series(index=series.index, dtype=float)
    for i in range(len(series)):
        w = series.iloc[:i].dropna()
        if len(w) < min_periods: out.iloc[i] = np.nan
        else:
            mu, sig = w.mean(), w.std()
            out.iloc[i] = (series.iloc[i] - mu) / sig if sig > 0 else 0.0
    return out
_regime['z_hpi']       = _expand_z(_regime['hpi_yoy'])
_regime['z_unemp_chg'] = _expand_z(_regime['unemp_yoy_chg'])
_regime['z_cpi']       = _expand_z(_regime['cpi_yoy'])
_regime['z_real_ff']   = _expand_z(_regime['real_fed_funds'])

def _vote(row):
    h, c, reasons = 0, 0, []
    if pd.notna(row['z_hpi']):
        if row['z_hpi'] > 1.5: h += 1; reasons.append(f'HPI+={row["z_hpi"]:+.1f}')
        elif row['z_hpi'] < -1.5: c += 1; reasons.append(f'HPI-={row["z_hpi"]:+.1f}')
    if pd.notna(row['z_unemp_chg']):
        if row['z_unemp_chg'] > 1.5: c += 1; reasons.append(f'Unemp+={row["z_unemp_chg"]:+.1f}')
        elif row['z_unemp_chg'] < -1.5: h += 1; reasons.append(f'Unemp-={row["z_unemp_chg"]:+.1f}')
    if pd.notna(row['z_cpi']):
        if row['z_cpi'] > 1.5: h += 1; reasons.append(f'CPI+={row["z_cpi"]:+.1f}')
        elif row['z_cpi'] < -1.5: c += 1; reasons.append(f'CPI-={row["z_cpi"]:+.1f}')
    if pd.notna(row['z_real_ff']):
        if row['z_real_ff'] > 1.5: h += 1; reasons.append(f'RealFF+={row["z_real_ff"]:+.1f}')
        elif row['z_real_ff'] < -1.5: c += 1; reasons.append(f'RealFF-={row["z_real_ff"]:+.1f}')
    if row['curve_inverted'] == 1: h += 1; reasons.append('curve_inv')
    return pd.Series({'h': h, 'c': c, 'reasons': ' | '.join(reasons)})
_votes = _regime.apply(_vote, axis=1)
_regime = pd.concat([_regime, _votes], axis=1)
def _classify(row):
    if row['h'] >= 3: return 'CYCLE TOP'
    if row['h'] >= 2: return 'LATE CYCLE'
    if row['c'] >= 3: return 'CYCLE BOTTOM'
    if row['c'] >= 2: return 'CONTRACTION'
    return 'MID CYCLE'
_regime['regime'] = _regime.apply(_classify, axis=1)

# ── Display current-year regime state ──
_cy = _regime[_regime['year'] == 2025]
if not _cy.empty:
    r = _cy.iloc[0]
    print('=' * 90)
    print(f'  REGIME for 2025: {r["regime"]}   (hot_votes={r["h"]}  cold_votes={r["c"]})')
    print(f'  Signals fired: {r["reasons"] if r["reasons"] else "none — all readings in normal range"}')
    print(f'  HPI YoY: {r["hpi_yoy"]*100:+.2f}%   Unemp chg: {r["unemp_yoy_chg"]:+.2f}pp   CPI YoY: {r["cpi_yoy"]:+.2f}%')
    print(f'  Real Fed funds: {r["real_fed_funds"]:+.2f}%   Yield curve (10y-2y): {r["yield_curve"]:+.2f}pp')
    print('=' * 90)
    print(f'  Investor interpretation:')
    print(f'    CYCLE TOP / LATE CYCLE → market elevated, mean reversion risk (bearish for new buys)')
    print(f'    MID CYCLE → no anomaly, trust trailing signals at face value')
    print(f'    CONTRACTION / CYCLE BOTTOM → stress in progress, cycle turning (bullish for new buys)')
    print(f'  This regime signal does NOT override picks — it provides context for your judgment.')
    print()

# ── Also show last few years for context ──
_recent = _regime[_regime['year'].between(2020, 2025)][['year','regime','h','c','reasons']].copy()
_recent = _recent.rename(columns={'h':'hot_votes', 'c':'cold_votes'})
print('Recent regime history:')
print(_recent.to_string(index=False))


## 8. Regime validation — per-year prediction vs actual

### Regime validation — did the detector's implicit prediction play out?

For each year, we compare the detector's regime label against realized 3-year forward HPI:
- **CYCLE TOP / LATE CYCLE** (bearish implication) → correct if forward 3yr HPI < 3.5%/yr (below trend)
- **MID CYCLE** (neutral implication) → correct if forward 3yr HPI within ±2pp of 4.5%/yr trend
- **CONTRACTION / CYCLE BOTTOM** (bullish implication) → correct if forward 3yr HPI > 5.5%/yr (above trend)

**Verdict color coding** in the table below:
- 🟢 GREEN = label correctly predicted forward direction
- 🟡 AMBER = partial match (right direction but muted magnitude)
- 🔴 RED = label got forward direction wrong (markets moved opposite to implication)
- ⚪ GRAY = forward 3yr not yet observable (2023+)

**Honest caveat**: markets can stay overbought/oversold longer than the label suggests. Extreme labels (CYCLE TOP / CYCLE BOTTOM) have historically underperformed at forward prediction because momentum overshoots. MID CYCLE labels are the most reliable.


In [ ]:
# Regime validation — historical accuracy per year
# Uses the _regime DataFrame already computed in Section 6.
# Adds forward 1yr and 3yr HPI, ground truth events, and verdict.

BENCHMARK_HPI = 4.5  # long-term national avg (%/yr)

# Forward HPI computation
_regime['hpi_fwd_1yr_pct'] = _regime['hpi_yoy'].shift(-1) * 100
def _fwd_3yr_ann(idx):
    if idx + 3 >= len(_regime): return np.nan
    yrs = _regime['hpi_yoy'].iloc[idx+1:idx+4].values
    if any(pd.isna(yrs)): return np.nan
    cum = np.prod([1 + y for y in yrs])
    return (cum ** (1/3) - 1) * 100
_regime['hpi_fwd_3yr_ann_pct'] = [_fwd_3yr_ann(i) for i in range(len(_regime))]

# Ground truth — known historical events
_GROUND_TRUTH = {
    1990:'S&L crisis / recession', 1991:'recession continuing', 1992:'jobless recovery',
    1993:'recovery', 1994:'Fed tightening cycle', 1995:'soft landing', 1996:'expansion',
    1997:'expansion', 1998:'LTCM / Asian crisis', 1999:'dot-com run-up',
    2000:'dot-com peak', 2001:'dot-com bust + 9/11', 2002:'jobless recovery',
    2003:'recovery, rates cut hard', 2004:'expansion, housing rising',
    2005:'SFR bubble peaking', 2006:'housing top', 2007:'crisis brewing',
    2008:'GFC CRASH', 2009:'depths of crash', 2010:'stimulus recovery',
    2011:'bottom + Euro crisis', 2012:'early recovery', 2013:'housing recovery',
    2014:'recovery continuing', 2015:'expansion', 2016:'expansion',
    2017:'expansion, Trump tax cuts', 2018:'Fed hiking cycle',
    2019:'curve inversion, late cycle', 2020:'COVID shock + stimulus',
    2021:'rent + HPI bubble', 2022:'rate shock (topping)',
    2023:'tightening peak', 2024:'tightening plateau', 2025:'rate normalization',
}
_regime['known_event'] = _regime['year'].map(_GROUND_TRUTH)

# Verdict — did label's implied direction play out?
def _verdict(row):
    fwd = row['hpi_fwd_3yr_ann_pct']
    if pd.isna(fwd): return 'n/a — fwd not observable'
    r = row['regime']
    if r in ('CYCLE TOP', 'LATE CYCLE'):
        if fwd < BENCHMARK_HPI - 1: return 'CORRECT'
        elif fwd > BENCHMARK_HPI + 1: return 'WRONG'
        else: return 'PARTIAL'
    elif r == 'MID CYCLE':
        if abs(fwd - BENCHMARK_HPI) <= 2: return 'CORRECT'
        elif fwd > BENCHMARK_HPI + 2: return 'PARTIAL'
        else: return 'PARTIAL'
    elif r in ('CONTRACTION', 'CYCLE BOTTOM'):
        if fwd > BENCHMARK_HPI + 1: return 'CORRECT'
        elif fwd < BENCHMARK_HPI - 1: return 'WRONG'
        else: return 'PARTIAL'
    return 'n/a'
_regime['verdict'] = _regime.apply(_verdict, axis=1)

# Build the validation DataFrame — years 2000+ (before that z-scores don't fire)
# Note: context cell stores vote counts as 'h' / 'c' — rename for readability
validation_df = _regime[_regime['year'] >= 2000][[
    'year', 'regime', 'h', 'c',
    'hpi_yoy', 'hpi_fwd_1yr_pct', 'hpi_fwd_3yr_ann_pct',
    'known_event', 'verdict'
]].copy()
validation_df['hpi_yoy'] = validation_df['hpi_yoy'] * 100
validation_df = validation_df.rename(columns={
    'hpi_yoy': 'hpi_yoy_%', 'h': 'hot_votes', 'c': 'cold_votes'
})
validation_df = validation_df.reset_index(drop=True)

# Color verdict column
def _color_verdict(v):
    if pd.isna(v): return ''
    if v == 'CORRECT': return 'background-color: #d4edda; color: #006600; font-weight: bold'
    if v == 'WRONG':   return 'background-color: #f8d7da; color: #b30000; font-weight: bold'
    if v == 'PARTIAL': return 'background-color: #fff3cd; color: #7a5a00'
    return 'background-color: #eeeeee; color: #666'  # n/a

def _color_regime(v):
    if v == 'CYCLE TOP':    return 'background-color: #f8d7da; font-weight: bold'
    if v == 'LATE CYCLE':   return 'background-color: #ffe8b3'
    if v == 'CYCLE BOTTOM': return 'background-color: #b3d9ff; font-weight: bold'
    if v == 'CONTRACTION':  return 'background-color: #cce5ff'
    return ''

validation_styled = (validation_df.style
    .format({'hpi_yoy_%': '{:+.2f}%', 'hpi_fwd_1yr_pct': '{:+.2f}%', 'hpi_fwd_3yr_ann_pct': '{:+.2f}%'}, na_rep='—')
    .map(_color_regime, subset=['regime'])
    .map(_color_verdict, subset=['verdict'])
    .set_caption('Regime label per year vs realized 3yr forward HPI — is the label\'s implicit prediction correct?')
)
validation_styled


In [ ]:
# Accuracy summary — how often did the regime label predict correctly?
scored = validation_df[validation_df['verdict'].isin(['CORRECT','WRONG','PARTIAL'])].copy()

# By regime state
by_regime = scored.groupby('regime').agg(
    n_years=('year', 'count'),
    n_correct=('verdict', lambda s: (s=='CORRECT').sum()),
    n_wrong=('verdict',   lambda s: (s=='WRONG').sum()),
    n_partial=('verdict', lambda s: (s=='PARTIAL').sum()),
).reset_index()
by_regime['accuracy_strict_%'] = (by_regime['n_correct'] / by_regime['n_years'] * 100).round(1)
by_regime['accuracy_incl_partial_%'] = ((by_regime['n_correct'] + by_regime['n_partial']) / by_regime['n_years'] * 100).round(1)

# Ordering
_order = ['CYCLE TOP','LATE CYCLE','MID CYCLE','CONTRACTION','CYCLE BOTTOM']
by_regime['_order'] = by_regime['regime'].map({r:i for i,r in enumerate(_order)})
by_regime = by_regime.sort_values('_order').drop(columns='_order').reset_index(drop=True)

print('ACCURACY BY REGIME STATE:')
print()
print(by_regime.to_string(index=False))
print()

# Overall
n_total = len(scored)
n_correct = (scored['verdict']=='CORRECT').sum()
n_wrong   = (scored['verdict']=='WRONG').sum()
n_partial = (scored['verdict']=='PARTIAL').sum()

overall = pd.DataFrame([
    {'metric': 'total years scored', 'value': n_total},
    {'metric': 'correct',            'value': int(n_correct)},
    {'metric': 'wrong',              'value': int(n_wrong)},
    {'metric': 'partial',            'value': int(n_partial)},
    {'metric': 'strict accuracy %',           'value': round(n_correct/n_total*100, 1) if n_total else 0},
    {'metric': 'including partial %',         'value': round((n_correct+n_partial)/n_total*100, 1) if n_total else 0},
])
print('OVERALL ACCURACY:')
print()
print(overall.to_string(index=False))
print()
print('Interpretation:')
print('  Extreme labels (CYCLE TOP / CYCLE BOTTOM) tend to be premature — markets stay overbought/oversold longer than the label suggests.')
print('  MID CYCLE is the most reliable label — high accuracy at predicting trend-level HPI.')
print('  Use extreme labels as CAUTION markers, not as directional forecasts.')


## 9. Regime forecasting tools — historical analog finder + Markov transition matrix

### Regime forecasting tools — analog finder + Markov transition matrix

**Neither of these is a forecast.** They are two different ways to convert current macro state into **historical base rates**:

1. **Analog finder** — "Which past years had macro state most similar to today? What happened next?" Uses Euclidean distance in 5-signal z-score space to identify the 5 nearest historical analogs, then reports their forward 1yr / 3yr / 5yr HPI distributions.

2. **Markov transition matrix** — "Given today's regime label, what's the empirical probability of each next-year label?" Computed from all historical year-to-year label transitions. Projects 1-year and 3-year forward label distributions.

**Why base rates, not forecasts?**
Professional forecasters (Fed SEP, IMF, Bloomberg consensus) have documented poor accuracy at 3-year macro forecasting — miss rates often worse than 50/50 for growth turning points. Base rates from historical analogs are honest: they show the range of outcomes conditional on similar starting states, without pretending to predict the specific path.

**How to use:**
- Wide analog dispersion (large min-max spread) = high uncertainty; plan for multiple scenarios.
- Tight analog dispersion = analogs consistently point one direction; higher confidence in the base case.
- Markov persistence probability high (>60%) = regime is sticky; expect current label to hold.
- Watch for high P(transition to CONTRACTION) or P(escalation to CYCLE TOP) as early scenario planning triggers.


In [ ]:
# ── Historical analog finder ────────────────────────────────────────────
# Find years where 5-signal macro state was most similar to today. Show what
# happened next as a base-rate distribution — NOT a forecast.

# 1. Add forward 5yr HPI (validation cell already added 1yr and 3yr)
def _fwd_5yr_ann(idx):
    if idx + 5 >= len(_regime): return np.nan
    yrs = _regime['hpi_yoy'].iloc[idx+1:idx+6].values
    if any(pd.isna(yrs)): return np.nan
    cum = np.prod([1 + y for y in yrs])
    return (cum ** (1/5) - 1) * 100
if 'hpi_fwd_5yr_ann_pct' not in _regime.columns:
    _regime['hpi_fwd_5yr_ann_pct'] = [_fwd_5yr_ann(i) for i in range(len(_regime))]

# 2. Build feature vector for each year (5-dim z-score state)
_FEATURES = ['z_hpi', 'z_unemp_chg', 'z_cpi', 'z_real_ff', 'curve_inverted']
_feat = _regime[_FEATURES].fillna(0.0)   # NaN z (early years) treated as neutral

# 3. Query = latest year with valid z-scores
_current_year = int(_regime['year'].max())
_query_row = _regime[_regime['year'] == _current_year].iloc[0]
_query_vec = _feat.iloc[_query_row.name].values

# 4. Euclidean distance to all OTHER years that have observable 3yr forward
_candidates = _regime[
    (_regime['year'] != _current_year) &
    (_regime['hpi_fwd_3yr_ann_pct'].notna())
].copy()
_cand_feat = _feat.loc[_candidates.index].values
_candidates['distance'] = np.sqrt(((_cand_feat - _query_vec) ** 2).sum(axis=1))

# 5. Top 5 nearest analogs
_analogs = _candidates.nsmallest(5, 'distance')[[
    'year', 'regime', 'h', 'c',
    'z_hpi', 'z_unemp_chg', 'z_cpi', 'z_real_ff', 'curve_inverted',
    'hpi_yoy', 'hpi_fwd_1yr_pct', 'hpi_fwd_3yr_ann_pct', 'hpi_fwd_5yr_ann_pct',
    'known_event', 'distance'
]].copy().reset_index(drop=True)
_analogs['hpi_yoy'] = _analogs['hpi_yoy'] * 100

print('=' * 110)
print(f'HISTORICAL ANALOG FINDER — 5 nearest years to current macro state')
print(f'Query year: {_current_year}  |  Label: {_query_row["regime"]}  |  Votes: {int(_query_row["h"])}H / {int(_query_row["c"])}C')
print('=' * 110)
print()
print('CURRENT STATE (z-scores + curve):')
for f in _FEATURES:
    v = _query_row.get(f, np.nan)
    if f == 'curve_inverted':
        print(f'  {f:<20}  {int(v) if pd.notna(v) else "—"}')
    else:
        print(f'  {f:<20}  {"—" if pd.isna(v) else f"{v:+.2f}"}')
print()
print('5 NEAREST HISTORICAL ANALOGS (Euclidean distance in z-score space, lower = more similar):')
print()

# Pretty display
def _color_verdict_dist(dist):
    if dist < 1.5: return 'background-color: #d4edda; font-weight: bold'
    if dist < 3.0: return 'background-color: #fff3cd'
    return 'background-color: #f8d7da'
def _color_fwd(v):
    if pd.isna(v): return ''
    if v < 0:      return 'background-color: #f8d7da; color: #b30000; font-weight: bold'
    if v < 3.5:    return 'background-color: #fff3cd; color: #7a5a00'
    if v <= 5.5:   return 'background-color: #d1ecf1; color: #0c5460'
    return 'background-color: #d4edda; color: #006600; font-weight: bold'

_analogs_styled = (_analogs.style
    .format({
        'z_hpi':'{:+.2f}', 'z_unemp_chg':'{:+.2f}', 'z_cpi':'{:+.2f}', 'z_real_ff':'{:+.2f}',
        'curve_inverted': '{:.0f}',
        'hpi_yoy':'{:+.2f}%',
        'hpi_fwd_1yr_pct':'{:+.2f}%', 'hpi_fwd_3yr_ann_pct':'{:+.2f}%', 'hpi_fwd_5yr_ann_pct':'{:+.2f}%',
        'distance':'{:.2f}',
    }, na_rep='—')
    .map(_color_verdict_dist, subset=['distance'])
    .map(_color_fwd, subset=['hpi_fwd_1yr_pct','hpi_fwd_3yr_ann_pct','hpi_fwd_5yr_ann_pct'])
    .set_caption(f'5 nearest historical analogs to {_current_year} — GREEN distance = very close match, AMBER = close, RED = weak match')
)
display(_analogs_styled)
print()

# 6. Base-rate distribution across analogs
print('=' * 110)
print('FORWARD-RETURN BASE RATE (across the 5 analogs)')
print('=' * 110)
print()

def _dist_summary(col, label):
    v = _analogs[col].dropna()
    if v.empty:
        print(f'  {label:<32}  no observable analogs')
        return
    n_neg    = int((v < 0).sum())
    n_below  = int(((v >= 0) & (v < 3.5)).sum())
    n_trend  = int(((v >= 3.5) & (v <= 5.5)).sum())
    n_above  = int((v > 5.5).sum())
    print(f'  {label:<32}  n={len(v)}   mean={v.mean():+.2f}%   median={v.median():+.2f}%   min={v.min():+.2f}%   max={v.max():+.2f}%')
    print(f'    P(negative)        {n_neg/len(v)*100:>3.0f}%   ({n_neg}/{len(v)} analogs)')
    print(f'    P(0 - 3.5%/yr)     {n_below/len(v)*100:>3.0f}%   ({n_below}/{len(v)}, below trend)')
    print(f'    P(3.5 - 5.5%/yr)   {n_trend/len(v)*100:>3.0f}%   ({n_trend}/{len(v)}, trend)')
    print(f'    P(>5.5%/yr)        {n_above/len(v)*100:>3.0f}%   ({n_above}/{len(v)}, above trend)')

_dist_summary('hpi_fwd_1yr_pct', 'Forward 1yr HPI'); print()
_dist_summary('hpi_fwd_3yr_ann_pct', 'Forward 3yr HPI (annualized)'); print()
_dist_summary('hpi_fwd_5yr_ann_pct', 'Forward 5yr HPI (annualized)'); print()

print('INTERPRETATION:')
print('  • These are NOT forecasts. They are historical base rates for years with similar macro state.')
print('  • Wide dispersion (large min-max spread) = high uncertainty — plan for multiple scenarios.')
print('  • Tight dispersion = analogs point one direction — higher confidence in base case.')
print('  • Cross-check the "known_event" column — are analog years fundamentally similar situations?')
print('  • Curve_inverted is 0/1 while z-scores span ±3; matching implicitly under-weights the curve.')


In [ ]:
# ── Regime transition Markov matrix ─────────────────────────────────────
# Empirical P(next_year_label | current_year_label) from full history.
# Projects 1-year and 3-year forward regime label distribution.

_regime_states = ['CYCLE TOP', 'LATE CYCLE', 'MID CYCLE', 'CONTRACTION', 'CYCLE BOTTOM']

# Build (current, next) transition pairs from the time series
_pairs = _regime[['year','regime']].copy()
_pairs['next_regime'] = _pairs['regime'].shift(-1)
_pairs = _pairs.dropna(subset=['next_regime'])

# Transition count matrix — reindex to enforce full state list on both axes
_transition_counts = pd.crosstab(_pairs['regime'], _pairs['next_regime'])
_transition_counts = _transition_counts.reindex(index=_regime_states, columns=_regime_states, fill_value=0)

# Probabilities (row-normalized)
_row_totals = _transition_counts.sum(axis=1)
_transition_probs = _transition_counts.div(_row_totals.replace(0, np.nan), axis=0)

print('=' * 100)
print('REGIME TRANSITION MATRIX (empirical, 1-year transitions)')
print('=' * 100)
print()
print(f'Sample: {len(_pairs)} year-to-year transitions observed ({int(_regime["year"].min())} to {int(_regime["year"].max())})')
print()

# Counts table
print('TRANSITION COUNTS (rows = current label, columns = next-year label):')
print()
_ct = _transition_counts.copy()
_ct['TOTAL'] = _row_totals.astype(int)
print(_ct.to_string())
print()

# Probabilities table with color coding
def _color_prob(v):
    if pd.isna(v) or v == 0: return 'color: #ccc'
    if v >= 0.60: return 'background-color: #d4edda; color: #006600; font-weight: bold'
    if v >= 0.30: return 'background-color: #d1ecf1; color: #0c5460'
    if v >= 0.10: return 'background-color: #fff3cd; color: #7a5a00'
    return 'color: #6c757d'

print('TRANSITION PROBABILITIES (row = current label, column = next-year label):')
print()
_probs_styled = (_transition_probs.fillna(0).style
    .format('{:.1%}')
    .map(_color_prob)
    .set_caption('P(next_year_label | current_year_label) — GREEN ≥60% (very sticky), BLUE 30-60% (likely), AMBER 10-30% (possible), gray <10% (rare)')
)
display(_probs_styled)
print()

# Current label + forward projection
_cur_label = _regime[_regime['year'] == _regime['year'].max()].iloc[0]['regime']
_cur_year  = int(_regime['year'].max())

print('=' * 100)
print(f'PROJECTED REGIME DISTRIBUTION FROM CURRENT STATE ({_cur_year} = {_cur_label})')
print('=' * 100)
print()

# 1-year forward
_p1 = _transition_probs.loc[_cur_label] if _cur_label in _transition_probs.index else pd.Series(dtype=float)
print(f'1 YEAR AHEAD ({_cur_year + 1}) — probability of each label:')
for state in _regime_states:
    p = (_p1.get(state, 0) or 0) * 100
    bar = '█' * int(round(p / 3))
    print(f'  {state:<15}  {p:5.1f}%  {bar}')
print()

# Multi-year forward via Markov iteration
def _project(current_label, n_years):
    if current_label not in _transition_probs.index:
        return pd.Series(dtype=float)
    v = pd.Series(0.0, index=_regime_states)
    v[current_label] = 1.0
    for _ in range(n_years):
        v = _transition_probs.T.dot(v).fillna(0)
    return v

_p3 = _project(_cur_label, 3)
print(f'3 YEARS AHEAD ({_cur_year + 3}) — probability of each label:')
for state in _regime_states:
    p = (_p3.get(state, 0) or 0) * 100
    bar = '█' * int(round(p / 3))
    print(f'  {state:<15}  {p:5.1f}%  {bar}')
print()

# Persistence stat
n_pairs = len(_pairs)
n_same  = int((_pairs['regime'] == _pairs['next_regime']).sum())
print('BASE-RATE INSIGHTS:')
print(f'  Total 1-year transitions observed: {n_pairs}')
print(f'  Same-label persistence: {n_same}/{n_pairs} = {n_same/n_pairs*100:.0f}%')
print(f'  → Regime labels are typically sticky year-over-year. Transitions to non-adjacent states are rare.')
print()

# Adjacent vs jump transitions
_adjacency = {s:i for i,s in enumerate(_regime_states)}
def _dist(pair):
    return abs(_adjacency[pair['regime']] - _adjacency[pair['next_regime']])
_pairs['jump'] = _pairs.apply(_dist, axis=1)
_n_stay   = int((_pairs['jump'] == 0).sum())
_n_1step  = int((_pairs['jump'] == 1).sum())
_n_2step  = int((_pairs['jump'] == 2).sum())
_n_3plus  = int((_pairs['jump'] >= 3).sum())
print('YEAR-OVER-YEAR TRANSITION DISTANCE:')
print(f'  Stay (same label):        {_n_stay:>3}/{n_pairs}  ({_n_stay/n_pairs*100:.0f}%)')
print(f'  Adjacent (1-step move):   {_n_1step:>3}/{n_pairs}  ({_n_1step/n_pairs*100:.0f}%)')
print(f'  2-step move:              {_n_2step:>3}/{n_pairs}  ({_n_2step/n_pairs*100:.0f}%)')
print(f'  3+ step move (rare):      {_n_3plus:>3}/{n_pairs}  ({_n_3plus/n_pairs*100:.0f}%)')
print()

# Small-sample warning
_smallest_n = int(_row_totals.min())
if _smallest_n < 5:
    _sparse = [_regime_states[i] for i, n in enumerate(_row_totals) if n < 5]
    print(f'⚠  Small-sample warning: {", ".join(_sparse)} have <5 historical observations each.')
    print(f'   Transition probabilities from these states are noisy estimates.')


## 10. Current year (2025) picks — actionable (5-year hold underwriting)

In [ ]:
# Current year (2025) picks — MF-native underwriting view (5-year hold)
CURRENT_YEAR       = 2025
RATE_STRESS_BP     = 0.005   # +50bp rate stress test
TARGET_ROE_ANNUAL  = 0.15    # target 15%/yr levered ROE
HORIZON            = 5
TARGET_ROE_5YR = (1 + TARGET_ROE_ANNUAL) ** HORIZON - 1

def levered_roe_5yr(noi_yield, g, mtg, ltv=LTV, horizon=HORIZON):
    '''Return cumulative 5yr levered ROE given entry NOI yield and annual NOI growth rate.'''
    debt_yield = ltv * mtg
    equity_pct = 1 - ltv
    annual_debt = debt_yield
    # Sum of compounding NOI yield over 5 years
    if g == 0:
        noi_sum = noi_yield * horizon
    else:
        noi_sum = noi_yield * ((1+g)*((1+g)**horizon - 1) / g)
    cumulative_cf = noi_sum - horizon * annual_debt
    capital_gain = (1+g)**horizon - 1
    return (cumulative_cf + capital_gain) / equity_pct

def solve_noi_growth(noi_yield, mtg, roe_target_5yr=0.0, ltv=LTV):
    '''Bisection: find annual NOI growth g such that 5yr levered ROE = target.'''
    lo, hi = -0.30, 0.40
    def f(g): return levered_roe_5yr(noi_yield, g, mtg, ltv) - roe_target_5yr
    if f(lo) > 0: return float('-inf')
    if f(hi) < 0: return float('inf')
    for _ in range(60):
        mid = (lo + hi) / 2
        if f(mid) < 0: lo = mid
        else: hi = mid
        if hi - lo < 1e-6: break
    return (lo + hi) / 2

picks = pick_for_year(CURRENT_YEAR)
mtg_base   = uni[uni['year'] == CURRENT_YEAR]['mortgage_rate'].iloc[0]
mtg_stress = mtg_base + RATE_STRESS_BP

print(f'=== {CURRENT_YEAR} picks — MF underwriting view (5-YEAR hold) ===')
print(f'FRED MORTGAGE30US at cohort year-end: {mtg_base*100:.2f}%   |   stress scenario: {mtg_stress*100:.2f}% (+50bp)')
print(f'Assumptions: 70% LTV interest-only 5yr hold, exit cap = entry cap, no transaction costs')
print(f'Target: {TARGET_ROE_ANNUAL*100:.0f}%/yr levered ROE ({TARGET_ROE_5YR*100:.1f}% cumulative over 5yr)')
print()

rows = []
for i in range(len(picks)):
    r = picks.iloc[i]
    cap  = r['cap_rate_proxy']
    opex = r['opex_ratio']
    zy   = r['zori_yoy'] if pd.notna(r['zori_yoy']) else 0.0
    noi_yld    = cap * (1 - opex)
    debt_yld   = LTV * mtg_base
    dscr       = noi_yld / debt_yld
    noi_gap    = noi_yld - debt_yld

    g_be_base   = solve_noi_growth(noi_yld, mtg_base,   roe_target_5yr=0.0)
    g_be_stress = solve_noi_growth(noi_yld, mtg_stress, roe_target_5yr=0.0)
    g_tgt_base  = solve_noi_growth(noi_yld, mtg_base,   roe_target_5yr=TARGET_ROE_5YR)

    # Projected 5yr levered ROE at trailing zori_yoy as NOI growth
    proj_roe_5yr = levered_roe_5yr(noi_yld, zy, mtg_base)
    proj_roe_ann = (1 + proj_roe_5yr) ** (1/HORIZON) - 1 if 1 + proj_roe_5yr > 0 else float('nan')

    rows.append({
        '#': i+1,
        'market':   f"{r['county_name'].replace(' County','')}, {r['state_name']}",
        'cap_gross':cap * 100,
        'opex%':    opex * 100,
        'noi_yld':  noi_yld * 100,
        'mtg':      mtg_base * 100,
        'debt_yld': debt_yld * 100,
        'DSCR':     dscr,
        'noi_gap':  noi_gap * 100,
        'zori_yoy': zy * 100,
        'NOI_grow_BE':      g_be_base * 100,
        'NOI_grow_BE_+50bp': g_be_stress * 100,
        'NOI_grow_TGT_15%':  g_tgt_base * 100,
        'Proj_ROE_@zori':   proj_roe_ann * 100 if pd.notna(proj_roe_ann) else None,
        'mf_bldgs_5to9':    int(r['mf_bldgs_5to9'])   if pd.notna(r.get('mf_bldgs_5to9'))   else None,
        'mf_bldgs_10to19':  int(r['mf_bldgs_10to19']) if pd.notna(r.get('mf_bldgs_10to19')) else None,
        'mf_bldgs_5to19':   int(r['mf_bldgs_5to19'])  if pd.notna(r.get('mf_bldgs_5to19'))  else None,
        'mf_bldgs_20to49':  int(r['mf_bldgs_20to49']) if pd.notna(r.get('mf_bldgs_20to49')) else None,
        'mf_bldgs_50plus':  int(r['mf_bldgs_50plus']) if pd.notna(r.get('mf_bldgs_50plus')) else None,
    })
df = pd.DataFrame(rows)

def _color_dscr(v):
    if pd.isna(v): return ''
    if v >= 1.20: return 'background-color: #d4edda; color: #006600; font-weight: bold'
    if v >= 1.00: return 'background-color: #fff3cd; color: #7a5a00'
    return 'background-color: #f8d7da; color: #b30000; font-weight: bold'
def _color_noi_gap(v):
    if pd.isna(v): return ''
    if v > 0: return 'color: #006600; font-weight: bold'
    return 'color: #b30000'
def _color_noi_growth(v):
    if pd.isna(v) or v == float('inf') or v == float('-inf'): return ''
    if v < 2: return 'background-color: #d4edda; color: #006600; font-weight: bold'
    if v < 4: return 'background-color: #fff3cd; color: #7a5a00'
    return 'background-color: #f8d7da; color: #b30000; font-weight: bold'
def _color_projected_roe(v):
    if pd.isna(v): return ''
    if v > TARGET_ROE_ANNUAL*100: return 'background-color: #d4edda; color: #006600; font-weight: bold'
    if v > 5: return 'background-color: #fff3cd; color: #7a5a00'
    return 'background-color: #f8d7da; color: #b30000; font-weight: bold'

styled = (df.style
    .format({
        'cap_gross':'{:.2f}%','opex%':'{:.0f}%','noi_yld':'{:.2f}%',
        'mtg':'{:.2f}%','debt_yld':'{:.2f}%','DSCR':'{:.2f}x',
        'noi_gap':'{:+.2f}pp','zori_yoy':'{:+.2f}%',
        'NOI_grow_BE':'{:+.2f}%/yr','NOI_grow_BE_+50bp':'{:+.2f}%/yr','NOI_grow_TGT_15%':'{:+.2f}%/yr',
        'Proj_ROE_@zori':'{:+.2f}%/yr',
        'mf_bldgs_5to9':'{:,.0f}','mf_bldgs_10to19':'{:,.0f}','mf_bldgs_5to19':'{:,.0f}',
        'mf_bldgs_20to49':'{:,.0f}','mf_bldgs_50plus':'{:,.0f}',
    }, na_rep='—')
    .map(_color_dscr, subset=['DSCR'])
    .map(_color_noi_gap, subset=['noi_gap'])
    .map(_color_noi_growth, subset=['NOI_grow_BE','NOI_grow_BE_+50bp','NOI_grow_TGT_15%'])
    .map(_color_projected_roe, subset=['Proj_ROE_@zori'])
    .set_caption(f'{CURRENT_YEAR} picks — 5-YEAR HOLD underwriting view')
)
display(styled)

print()
print('COLUMN GUIDE:')
print('  cap_gross          = gross ACS cap-rate proxy')
print('  noi_yld            = net NOI yield after opex')
print('  DSCR               = noi_yld / (LTV × mortgage rate). Green ≥1.20, Amber 1.00-1.20, Red <1.00')
print('  noi_gap            = net cash carry vs debt service (positive = self-sustaining)')
print('  zori_yoy           = trailing 12mo rent change')
print('  NOI_grow_BE        = annual NOI growth needed for 5yr levered ROE = 0 (breakeven)')
print('  NOI_grow_BE_+50bp  = same, at mortgage +50bp (rate stress)')
print('  NOI_grow_TGT_15%   = annual NOI growth needed for 15%/yr levered ROE over 5yr')
print('  Proj_ROE_@zori     = projected annualized levered ROE if trailing zori_yoy holds for 5yr')
print()
print('READING:')
print('  • NOI_grow_BE < 3%/yr is defensible at trend rent growth')
print('  • NOI_grow_TGT_15% > 6%/yr means you need above-trend growth to hit target')
print('  • Proj_ROE_@zori > 15%/yr = deal projected to hit target on trailing momentum')
